# 02 · Data Preprocessing
Independent Colab notebook. Upload the original Kaggle CSV when prompted. We create a stratified 80/20 split (random_state 42); preprocessing is fitted only inside training pipelines to prevent leakage. No SMOTE is used on test data.


In [ ]:
%pip -q install pandas numpy scikit-learn
from pathlib import Path
import pandas as pd, numpy as np, json
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
try:
 from google.colab import drive,files
 drive.mount('/content/drive')
except ImportError: files=None
ROOT=Path('/content/drive/MyDrive/insurance_fraud_ml'); (ROOT/'data').mkdir(parents=True,exist_ok=True)
paths=[p for p in (ROOT/'data').glob('*.csv') if p.name not in ['X_train.csv','X_test.csv','y_train.csv','y_test.csv']]
if paths: csv=paths[0]
elif files:
 print('Upload original CSV'); up=files.upload()
 if not up: raise FileNotFoundError('Upload CSV')
 csv=Path(next(iter(up)))
else: csv=Path(input('CSV path: '))
df=pd.read_csv(csv); target='FraudFound_P'
if target not in df: raise ValueError('Expected FraudFound_P target')
df[target]=pd.to_numeric(df[target],errors='coerce'); df=df.dropna(subset=[target]); df[target]=df[target].astype(int)
if not set(df[target].unique()).issubset({0,1}): raise ValueError('Target must be 0/1')
X=df.drop(columns=[target]); y=df[target]
ids=[c for c in X if any(t in c.lower() for t in ['policy_number','claim_id','record_id','customer_id'])]; X=X.drop(columns=ids)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
nums=Xtr.select_dtypes(include='number').columns.tolist(); cats=Xtr.select_dtypes(exclude='number').columns.tolist()
def makeprep(scale=False):
 n=Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True))]+([('scale',StandardScaler())] if scale else []))
 c=Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',min_frequency=2))])
 return ColumnTransformer([('numeric',n,nums),('categorical',c,cats)],verbose_feature_names_out=False)
print('rows',len(df),'train/test',Xtr.shape,Xte.shape,'fraud rates',ytr.mean(),yte.mean(),'IDs removed',ids)
for name,obj in [('X_train',Xtr),('X_test',Xte),('y_train',ytr),('y_test',yte)]: obj.to_csv(ROOT/'data'/f'{name}.csv',index=False)
meta={'target':target,'random_state':42,'test_size':.2,'removed_identifiers':ids,'numeric_features':nums,'categorical_features':cats}; (ROOT/'data/preprocessing_metadata.json').write_text(json.dumps(meta,indent=2))

Numeric missing values use median imputation; categorical values use the most frequent category and one-hot encoding. Scaling is included only when the downstream model needs distances/feature scale. Named identifiers are removed to discourage memorization; printed list should be reviewed. Saved splits are reproducible artifacts, not modified raw data.